# Business Impact — From Baseline to Matrix Factorization

This notebook translates model metric improvements into **concrete business value**.

Raw numbers like RMSE=0.85 are hard to act on. This notebook answers:
- What does each model improvement mean for a real streaming platform?
- How do Precision@10 and Recall@10 map to user behaviour and revenue?
- Where does the biggest business gain come from?

---

**Model progression:**

| Model | What it knows |
|---|---|
| Global Mean | Nothing — same prediction for everyone |
| Item Mean | Movie quality only |
| User Mean | User strictness only |
| User + Item Bias | Both user strictness and movie quality |
| Matrix Factorization | Personalised taste via latent factors |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

BG, PANEL, GRID_C = "#0f0f14", "#16161f", "#2a2a35"
TEXT_C, ACCENT, ACCENT2, ACCENT3, ACCENT4 = "#d4d4e0", "#7DF9C4", "#F97D7D", "#7DA8F9", "#F9E07D"

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=TEXT_C, fontsize=10, pad=9, fontweight="bold")
    ax.tick_params(colors=TEXT_C, labelsize=8)
    ax.xaxis.label.set_color(TEXT_C)
    ax.yaxis.label.set_color(TEXT_C)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_C)
    ax.grid(color=GRID_C, linewidth=0.5, linestyle="--", alpha=0.7)

## 1. Metric Summary Across All Models

In [ ]:
# Results collected from baselines.ipynb, collaborative.ipynb
models = [
    {"name": "Global Mean",              "rmse": 1.0183, "mae": 0.8116, "precision": 0.0070, "recall": 0.0040, "f1": 0.0052},
    {"name": "Item Mean",                "rmse": 0.9351, "mae": 0.7246, "precision": 0.0180, "recall": 0.0110, "f1": 0.0137},
    {"name": "User Mean",                "rmse": 0.9287, "mae": 0.7216, "precision": 0.0090, "recall": 0.0055, "f1": 0.0068},
    {"name": "User + Item Bias",         "rmse": 0.8573, "mae": 0.6564, "precision": 0.0220, "recall": 0.0140, "f1": 0.0171},
    {"name": "Matrix Factorization",     "rmse": 0.8236, "mae": 0.6317, "precision": 0.0263, "recall": 0.0237, "f1": 0.0249},
]

df = pd.DataFrame(models)

print("=" * 80)
print(f"  {'Model':<24} {'RMSE':>8}  {'MAE':>8}  {'P@10':>8}  {'R@10':>8}  {'F1@10':>8}")
print("=" * 80)
for _, row in df.iterrows():
    print(f"  {row['name']:<24} {row['rmse']:>8.4f}  {row['mae']:>8.4f}  "
          f"{row['precision']:>8.4f}  {row['recall']:>8.4f}  {row['f1']:>8.4f}")
print("=" * 80)

## 2. What Does Each Improvement Mean for the Business?

### RMSE — Rating Prediction Accuracy

RMSE measures how far off our predicted rating is from the actual rating.

| Improvement | RMSE drop | Business meaning |
|---|---|---|
| Global Mean → Item Mean | 1.018 → 0.935 | System now knows *Shawshank* is better than *Anaconda* — stops recommending universally bad movies |
| Item Mean → User + Item Bias | 0.935 → 0.857 | System now knows harsh vs generous raters — stops over-predicting for users who always rate low |
| User + Item Bias → MF | 0.857 → 0.824 | System now understands *taste* — a sci-fi fan gets sci-fi recommendations, not just popular movies |

### Precision@10 — Are the Top 10 Recommendations Good?

Precision@10 = fraction of top-10 recommended movies the user actually likes (rated ≥4).

| Model | P@10 | Out of 10 recommendations... |
|---|---|---|
| Global Mean | 0.007 | 0 good recommendations |
| Item Mean | 0.018 | ~0.2 good recommendations |
| User + Item Bias | 0.022 | ~0.2 good recommendations |
| **Matrix Factorization** | **0.026** | **~0.3 good recommendations** |

### Recall@10 — Are We Finding the Movies the User Would Love?

Recall@10 = fraction of movies the user would love that actually appear in our top-10 list.

MF recall (0.024) is nearly **6× higher** than Global Mean (0.004) — meaning MF surfaces far more of the movies a user would genuinely enjoy.

## 3. Business KPI Translation

Assuming a platform with **1 million users**, **10 recommendations per session**, and a **click-through rate proportional to Precision@10**:

In [ ]:
N_USERS          = 1_000_000   # platform scale
RECS_PER_SESSION = 10
REVENUE_PER_CLICK = 0.50       # $ revenue per engaged click (watch, add to list)
SESSIONS_PER_MONTH = 8         # avg sessions per user per month
CHURN_REDUCTION_PER_P10 = 0.5  # % churn reduction per 0.01 P@10 improvement (industry estimate)
MONTHLY_REVENUE_PER_USER = 12  # $ subscription revenue per user per month

print("=" * 72)
print(f"  {'Model':<24} {'P@10':>6}  {'Clicks/session':>14}  {'Monthly clicks':>14}")
print("=" * 72)

baseline_clicks = None
for m in models:
    clicks_per_session = m["precision"] * RECS_PER_SESSION
    monthly_clicks     = clicks_per_session * SESSIONS_PER_MONTH * N_USERS
    if baseline_clicks is None:
        baseline_clicks = monthly_clicks
    print(f"  {m['name']:<24} {m['precision']:>6.4f}  {clicks_per_session:>14.3f}  {monthly_clicks:>14,.0f}")

print("=" * 72)

# Lift from Global Mean → MF
mf_precision      = models[-1]["precision"]
baseline_precision = models[0]["precision"]

mf_clicks_monthly      = mf_precision * RECS_PER_SESSION * SESSIONS_PER_MONTH * N_USERS
baseline_clicks_monthly = baseline_precision * RECS_PER_SESSION * SESSIONS_PER_MONTH * N_USERS
extra_clicks            = mf_clicks_monthly - baseline_clicks_monthly
extra_revenue           = extra_clicks * REVENUE_PER_CLICK

print(f"""
Business impact of moving from Global Mean → Matrix Factorization:

  Extra clicks/month        : {extra_clicks:>12,.0f}
  Extra revenue/month (est.): ${extra_revenue:>11,.0f}
  Extra revenue/year  (est.): ${extra_revenue * 12:>11,.0f}
""")

# Churn reduction
p10_improvement    = mf_precision - baseline_precision
churn_reduction_pct = (p10_improvement / 0.01) * CHURN_REDUCTION_PER_P10
users_retained      = N_USERS * (churn_reduction_pct / 100)
retention_revenue   = users_retained * MONTHLY_REVENUE_PER_USER * 12

print(f"""Churn reduction from better recommendations:

  P@10 improvement          : {p10_improvement:>12.4f}
  Estimated churn reduction : {churn_reduction_pct:>11.2f}%
  Users retained            : {users_retained:>12,.0f}
  Retention revenue/year    : ${retention_revenue:>11,.0f}
""")

## 4. Where Does Each Model Fail the Business?

In [ ]:
failures = {
    "Global Mean": [
        "Recommends the same movies to everyone",
        "Zero personalisation — ignores user taste entirely",
        "Users churn because recommendations feel irrelevant",
        "No way to promote new content strategically",
    ],
    "Item Mean": [
        "Only recommends globally popular movies (popularity bias)",
        "Niche users (horror fans, documentary lovers) are ignored",
        "New movies with few ratings are never surfaced (cold start)",
        "Same top-10 for every user — no differentiation",
    ],
    "User Mean": [
        "Knows the user is generous/harsh but not what they like",
        "Still recommends same movies to all users",
        "Cannot distinguish between genres or content types",
    ],
    "User + Item Bias": [
        "No taste modelling — can't distinguish sci-fi fan from rom-com fan",
        "Recommendations converge to the same high-bias movies for all users",
        "Misses serendipitous discoveries (niche gems the user would love)",
    ],
    "Matrix Factorization": [
        "Cold start — struggles with new users or new movies",
        "Popularity bias still present in latent factors",
        "Does not use content signals (genre, cast, director)",
        "Static model — does not adapt to changing user taste over time",
    ],
}

for model, issues in failures.items():
    print(f"\n{model}")
    print("-" * len(model))
    for issue in issues:
        print(f"  ✗  {issue}")

## 5. Visualisation — Business Impact

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

names      = [m["name"].replace(" + ", "\n+").replace(" Bias", "\nBias") for m in models]
rmses      = [m["rmse"]      for m in models]
precisions = [m["precision"] for m in models]
recalls    = [m["recall"]    for m in models]
f1s        = [m["f1"]        for m in models]
colors     = [ACCENT2, ACCENT3, ACCENT4, ACCENT3, ACCENT]
colors[-1] = ACCENT   # highlight MF

# (a) RMSE progression
ax = fig.add_subplot(gs[0, 0])
bars = ax.bar(range(len(models)), rmses, color=colors, edgecolor=BG, width=0.6)
for b, v in zip(bars, rmses):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.4f}",
            ha="center", va="bottom", color=TEXT_C, fontsize=7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=6.5)
ax.set_ylabel("RMSE (lower = better)")
style_ax(ax, "Rating Prediction Error (RMSE)")

# (b) Precision@10 progression
ax = fig.add_subplot(gs[0, 1])
bars = ax.bar(range(len(models)), precisions, color=colors, edgecolor=BG, width=0.6)
for b, v in zip(bars, precisions):
    ax.text(b.get_x() + b.get_width()/2, v + 0.0003, f"{v:.4f}",
            ha="center", va="bottom", color=TEXT_C, fontsize=7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=6.5)
ax.set_ylabel("Precision@10 (higher = better)")
style_ax(ax, "Recommendation Quality (Precision@10)")

# (c) Recall@10 progression
ax = fig.add_subplot(gs[0, 2])
bars = ax.bar(range(len(models)), recalls, color=colors, edgecolor=BG, width=0.6)
for b, v in zip(bars, recalls):
    ax.text(b.get_x() + b.get_width()/2, v + 0.0003, f"{v:.4f}",
            ha="center", va="bottom", color=TEXT_C, fontsize=7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=6.5)
ax.set_ylabel("Recall@10 (higher = better)")
style_ax(ax, "Content Discovery (Recall@10)")

# (d) Monthly clicks at 1M users
ax = fig.add_subplot(gs[1, 0])
monthly_clicks = [m["precision"] * RECS_PER_SESSION * SESSIONS_PER_MONTH * N_USERS / 1e6 for m in models]
bars = ax.bar(range(len(models)), monthly_clicks, color=colors, edgecolor=BG, width=0.6)
for b, v in zip(bars, monthly_clicks):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.2f}M",
            ha="center", va="bottom", color=TEXT_C, fontsize=7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=6.5)
ax.set_ylabel("Monthly clicks (millions)")
style_ax(ax, "Est. Monthly Clicks (1M users)")

# (e) RMSE reduction waterfall
ax = fig.add_subplot(gs[1, 1])
improvements = [0] + [rmses[i] - rmses[i+1] for i in range(len(rmses)-1)]
cumulative   = [rmses[0] - r for r in rmses]
bars = ax.bar(range(len(models)), cumulative, color=colors, edgecolor=BG, width=0.6)
for b, v in zip(bars, cumulative):
    ax.text(b.get_x() + b.get_width()/2, v + 0.002, f"{v:.4f}",
            ha="center", va="bottom", color=TEXT_C, fontsize=7)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(names, fontsize=6.5)
ax.set_ylabel("Cumulative RMSE reduction")
style_ax(ax, "Cumulative RMSE Improvement vs Global Mean")

# (f) P@10 vs Recall@10 scatter
ax = fig.add_subplot(gs[1, 2])
for i, m in enumerate(models):
    ax.scatter(m["precision"], m["recall"], color=colors[i], s=120, zorder=5)
    ax.annotate(m["name"], (m["precision"], m["recall"]),
                textcoords="offset points", xytext=(6, 4),
                color=TEXT_C, fontsize=7)
ax.set_xlabel("Precision@10")
ax.set_ylabel("Recall@10")
style_ax(ax, "Precision vs Recall Trade-off")

fig.suptitle("Business Impact — Recommendation Model Progression",
             color=TEXT_C, fontsize=13, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()

## 6. Key Business Takeaways

### Biggest single gain: Item Mean → awareness of movie quality
Moving from Global Mean to Item Mean gives the largest RMSE drop (0.083). The system stops recommending objectively bad movies. This is the **cheapest win** — just compute average ratings per movie.

### Biggest personalisation gain: User + Item Bias → Matrix Factorization
MF nearly **doubles Recall@10** (0.014 → 0.024) over User+Item Bias. This means the system surfaces twice as many movies the user would genuinely love — directly impacting **watch time and session length**.

### What the metrics don't capture
| Business metric | Not measured here | Why it matters |
|---|---|---|
| Serendipity | Did we recommend something surprising the user loved? | Drives word-of-mouth and engagement |
| Diversity | Are we recommending only one genre? | Prevents filter bubbles, increases breadth of engagement |
| Novelty | Are we recommending things the user hasn't seen before? | Reduces recommendation fatigue |
| Coverage | What % of the catalogue gets recommended? | Important for content licensing ROI |
| Latency | How fast can we serve recommendations? | Critical for production — a slow model loses users |

### Next steps to maximise business value
1. **Hybrid model** — combine MF (personalisation) with content-based (genre/metadata) to handle cold start
2. **Temporal signals** — weight recent ratings more heavily to capture drift in taste
3. **Implicit feedback** — use watch time, re-watches, and search queries as additional signals
4. **A/B test** — deploy MF vs User+Item Bias to a subset of users and measure actual CTR, watch time, and churn